# Capstone 03 - Day 2 - Silver layer (clean rebuild)

Run the cells **top to bottom, once**, on a blank `silver_schema`. This notebook replaces tasks 4.1 to 8.4 of your earlier version.

**What changed compared with the earlier notebook**
1. `silver_flight_current` now has **one row per flight (11,500)**, not only the 520 flights that had a status change.
2. CDC is applied with a guarded Delta **MERGE** (an older update can never overwrite a newer status).
3. The 40 removed duplicates are **audited** in `silver_rejected_records` (rule `DQ08`), so `11,540 = 11,500 + 40` is provable from tables.
4. Every rule in the catalogue is implemented (`is_fare_outlier`, reference checks, UTC assumption flag, etc.).
5. `silver_status_history` starts with each flight's original status, so the as-of join works.
6. The tests call the **same functions** the pipeline uses, so they fail if the pipeline is broken.

**Expected results on your data**

| Item | Expected |
|---|---|
| Bronze flight rows to Silver flights | 11,540 to 11,500 |
| Rejected (EVENTS, rule DQ08 duplicates) | 40 |
| Fare outliers (above 5,000) / blank flight_type / timezone not valid | 35 / 12 / 11,500 |
| `silver_flight_current` rows | 11,500 (EVENT 10,980, CDC 520) |
| Current status mix | CANCELLED 171, COMPLETED 2,890, FAILED 2,836, OPEN 2,740, PENDING 2,690, REVERSED 173 |
| `silver_status_history` rows | 12,020 (11,500 initial + 520 changes) |
| `silver_secondary_events` | 310 |
| `silver_flight_event_status_asof` | 11,500 |

In [0]:
# =====================================================
# CAPSTONE 03 - DAY 2 : SILVER LAYER CONFIGURATION
# =====================================================
import zoneinfo
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = 'capstone_3flight'
BRONZE_SCHEMA = 'flight_schema'
SILVER_SCHEMA = 'silver_schema'

# True = drop the Silver tables this notebook creates, so everything is rebuilt from a blank schema.
RESET_SILVER = True

# Bronze source tables
BRONZE_FLIGHT_TABLE = f'{CATALOG}.{BRONZE_SCHEMA}.bronze_flight_events'
BRONZE_CDC_TABLE = f'{CATALOG}.{BRONZE_SCHEMA}.bronze_status_cdc'
BRONZE_SECONDARY_TABLE = f'{CATALOG}.{BRONZE_SCHEMA}.bronze_secondary_events'

# Reference tables
AIRLINE_TABLE = f'{CATALOG}.{BRONZE_SCHEMA}.airline'
AIRPORT_TABLE = f'{CATALOG}.{BRONZE_SCHEMA}.airport'

# Silver target tables
SILVER_FLIGHT_TABLE = f'{CATALOG}.{SILVER_SCHEMA}.silver_flight_events'
SILVER_REJECTED_TABLE = f'{CATALOG}.{SILVER_SCHEMA}.silver_rejected_records'
SILVER_STATUS_HISTORY = f'{CATALOG}.{SILVER_SCHEMA}.silver_status_history'
SILVER_FLIGHT_CURRENT = f'{CATALOG}.{SILVER_SCHEMA}.silver_flight_current'
SILVER_SECONDARY_TABLE = f'{CATALOG}.{SILVER_SCHEMA}.silver_secondary_events'
SILVER_ASOF_TABLE = f'{CATALOG}.{SILVER_SCHEMA}.silver_flight_event_status_asof'
DQ_TABLE = f'{CATALOG}.{SILVER_SCHEMA}.dq_rule_catalogue'

ALL_SILVER_TABLES = [SILVER_FLIGHT_TABLE, SILVER_REJECTED_TABLE, SILVER_STATUS_HISTORY, SILVER_FLIGHT_CURRENT,
                     SILVER_SECONDARY_TABLE, SILVER_ASOF_TABLE, DQ_TABLE]

spark.conf.set('spark.sql.session.timeZone', 'UTC')      # every timestamp in this project is handled as UTC
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}')

if RESET_SILVER:
    for tbl in ALL_SILVER_TABLES + [f'{CATALOG}.{SILVER_SCHEMA}.tmp_test_current', f'{CATALOG}.{SILVER_SCHEMA}.tmp_test_history']:
        spark.sql(f'DROP TABLE IF EXISTS {tbl}')
    print('Silver tables dropped - starting from a blank schema.')

# Verify the Bronze sources exist and have the expected row counts
expected_bronze = {BRONZE_FLIGHT_TABLE: 11540, BRONZE_CDC_TABLE: 520, BRONZE_SECONDARY_TABLE: 310,
                   AIRLINE_TABLE: 110, AIRPORT_TABLE: 150}
for table, expected in expected_bronze.items():
    actual = spark.table(table).count()
    print(table, ':', actual, 'rows', '(OK)' if actual == expected else f'(EXPECTED {expected})')
    assert actual == expected, f'{table} has {actual} rows, expected {expected}'
print('Silver configuration verified.')

In [0]:
# =====================================================
# DATA QUALITY RULE CATALOGUE  (saved as a table)
# =====================================================
dq_rules = [
    # rule_id, stage, column_name, validation_condition, severity, expected_action, description
    ('DQ01', 'EVENTS', 'flight_id', 'Must be FLIGHT- followed by 7 digits (not NULL or blank)', 'CRITICAL', 'QUARANTINE', 'Every flight event needs a valid business key'),
    ('DQ02', 'EVENTS', 'event_timestamp', 'Must be a valid timestamp', 'CRITICAL', 'QUARANTINE', 'Event time is mandatory'),
    ('DQ11', 'EVENTS', 'ingest_timestamp', 'Must be a valid timestamp', 'CRITICAL', 'QUARANTINE', 'Ingest time is mandatory'),
    ('DQ03', 'EVENTS', 'fare_amount', 'Must be numeric and not negative', 'HIGH', 'QUARANTINE', 'Fare must be usable downstream'),
    ('DQ09', 'EVENTS', 'delay_minutes', 'Must be an integer and not negative', 'HIGH', 'QUARANTINE', 'Delay is a KPI input'),
    ('DQ10', 'EVENTS', 'flight_status', 'Must be OPEN, PENDING, COMPLETED or FAILED', 'HIGH', 'QUARANTINE', 'Unknown statuses cannot be trusted'),
    ('DQ06', 'EVENTS', 'airline_id', 'Must match the airline reference table', 'HIGH', 'QUARANTINE', 'Prevent unexplained airline mismatches'),
    ('DQ07', 'EVENTS', 'airport_id', 'Must match the airport reference table', 'HIGH', 'QUARANTINE', 'Prevent unexplained airport mismatches'),
    ('DQ08', 'EVENTS', 'flight_id', 'Exact duplicate: same flight_id and identical 12 source columns', 'MEDIUM', 'DEDUPLICATE', 'Keep copy from the earliest file; audit the removed copy in the rejected table'),
    ('DQ12', 'EVENTS', 'flight_id', 'Same flight_id with different content', 'MEDIUM', 'DEDUPLICATE', 'Newest ingest_timestamp wins; audit the older record'),
    ('DQ04', 'EVENTS', 'flight_type', 'Optional: blank is permitted', 'MEDIUM', 'FLAG', 'Fill UNKNOWN and set flight_type_was_missing'),
    ('DQ05', 'EVENTS', 'fare_amount', 'Above 5000 is an extreme value', 'MEDIUM', 'FLAG', 'Keep the row; set is_fare_outlier (threshold needs client approval)'),
    ('DQ14', 'EVENTS', 'delay_minutes', 'Above 180 minutes is an extreme value', 'MEDIUM', 'FLAG', 'Keep the row; set is_delay_outlier'),
    ('DQ13', 'EVENTS', 'timezone', 'Must be a real time-zone name', 'MEDIUM', 'FLAG', 'Otherwise assume UTC and set timezone_is_valid = false'),
    ('DQC1', 'CDC', 'flight_id', 'Must not be NULL or blank', 'CRITICAL', 'QUARANTINE', 'A status change needs a flight'),
    ('DQC2', 'CDC', 'change_timestamp', 'Must be a valid timestamp', 'CRITICAL', 'QUARANTINE', 'Needed to order changes'),
    ('DQC3', 'CDC', 'source_sequence', 'Must be an integer', 'CRITICAL', 'QUARANTINE', 'Needed to order changes'),
    ('DQC4', 'CDC', 'new_status', 'Must be COMPLETED, REVERSED or CANCELLED', 'HIGH', 'QUARANTINE', 'Unknown target status'),
    ('DQC5', 'CDC', 'flight_id', 'Must exist in silver_flight_events', 'HIGH', 'QUARANTINE', 'Orphan status update'),
    ('DQC6', 'CDC', 'flight_id + source_sequence', 'Must be unique', 'MEDIUM', 'DEDUPLICATE', 'Repeated sequence would break history'),
    ('DQS1', 'SECONDARY', 'event_id', 'Must not be NULL or blank', 'CRITICAL', 'QUARANTINE', 'Event key'),
    ('DQS2', 'SECONDARY', 'flight_id', 'Must exist in silver_flight_events', 'HIGH', 'QUARANTINE', 'Orphan secondary event'),
    ('DQS3', 'SECONDARY', 'event_id', 'Must be unique', 'MEDIUM', 'DEDUPLICATE', 'Keep earliest copy'),
]
dq_columns = ['rule_id', 'stage', 'column_name', 'validation_condition', 'severity', 'expected_action', 'description']
dq_df = spark.createDataFrame(dq_rules, dq_columns)

(dq_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(DQ_TABLE))
print('DQ catalogue saved:', DQ_TABLE, '| rules:', spark.table(DQ_TABLE).count())
display(spark.table(DQ_TABLE).orderBy('stage', 'rule_id'))

In [0]:
# =====================================================
# BRONZE PROFILE (read-only evidence)
# =====================================================
flight_bronze_df = spark.table(BRONZE_FLIGHT_TABLE)
print('Total Bronze records:', flight_bronze_df.count())
flight_bronze_df.printSchema()

rescued = flight_bronze_df.filter(F.col('_rescued_data').isNotNull()).count() if '_rescued_data' in flight_bronze_df.columns else 0
print('Rescued-data rows:', rescued)

print('NULL / blank profile')
checked = ['flight_id', 'event_timestamp', 'ingest_timestamp', 'airline_id', 'airport_id', 'flight_type',
           'flight_status', 'fare_amount', 'delay_minutes']
display(flight_bronze_df.agg(*[F.count(F.when(F.col(c).isNull() | (F.col(c).cast('string') == ''), 1)).alias(c) for c in checked]))

print('Status distribution')
display(flight_bronze_df.groupBy('flight_status').count().orderBy('flight_status'))
print('Flight type distribution')
display(flight_bronze_df.groupBy('flight_type').count().orderBy('flight_type'))
print('Fare statistics')
display(flight_bronze_df.select(F.min('fare_amount').alias('min_fare'), F.max('fare_amount').alias('max_fare'),
                                F.avg('fare_amount').alias('avg_fare'), F.expr('percentile_approx(fare_amount, 0.5)').alias('median_fare')))

SOURCE_COLUMNS = ['flight_id', 'event_timestamp', 'ingest_timestamp', 'airline_id', 'airport_id', 'flight_type',
                  'flight_status', 'timezone', 'fare_amount', 'delay_minutes', 'aircraft_id', 'source_system']

exact_dup_rows = flight_bronze_df.count() - flight_bronze_df.select(*SOURCE_COLUMNS).distinct().count()
distinct_ids = flight_bronze_df.select('flight_id').distinct().count()
print('Exact duplicate rows (12 source columns):', exact_dup_rows)
print('Distinct flight_id:', distinct_ids, '| rows - distinct ids =', flight_bronze_df.count() - distinct_ids)
print('Fares above 5000:', flight_bronze_df.filter(F.col('fare_amount') > 5000).count())
print('Distinct statuses:', flight_bronze_df.select('flight_status').distinct().count(), '(CANCELLED does not exist in flight events)')
print('Distinct timezone values:', [r[0] for r in flight_bronze_df.select('timezone').distinct().collect()])

assert exact_dup_rows == 40 and distinct_ids == 11500

In [0]:
# =====================================================
# SILVER FUNCTIONS  (the pipeline AND the tests use these)
# =====================================================
ALLOWED_STATUS = ['OPEN', 'PENDING', 'COMPLETED', 'FAILED']
CDC_ALLOWED_STATUS = ['COMPLETED', 'REVERSED', 'CANCELLED']
FARE_OUTLIER_THRESHOLD = 5000.0       # normal fares top out at 1,099.99; the 35 injected ones start at 8,110.50
DELAY_OUTLIER_THRESHOLD = 180         # minutes; nothing in this data exceeds 90
DEFAULT_SOURCE_TZ = 'UTC'             # assumption: the timezone column holds USD, which is not a time zone
VALID_TZ = sorted(zoneinfo.available_timezones())


def _clean_text(df, cols, upper=False):
    '''Trim text; blank becomes NULL; optionally upper-case.'''
    for c in cols:
        e = F.trim(F.col(c))
        if upper:
            e = F.upper(e)
        df = df.withColumn(c, F.nullif(e, F.lit('')))
    return df


def _first_failing_rule(df, checks):
    '''checks = [(rule_id, reason, bad_condition)]. The FIRST failing rule wins, so a row is rejected once.'''
    rule = F.lit(None).cast('string')
    reason = F.lit(None).cast('string')
    for rid, why, cond in reversed(checks):
        rule = F.when(cond, F.lit(rid)).otherwise(rule)
        reason = F.when(cond, F.lit(why)).otherwise(reason)
    return df.withColumn('reject_rule', rule).withColumn('reject_reason', reason)


# ---------------- flight events ----------------
def classify_events(bronze_df, airline_keys, airport_keys):
    d = bronze_df
    if 'source_schema_version' not in d.columns:
        d = d.withColumn('source_schema_version', F.lit(None).cast('string'))
    d = d.withColumn('flight_type_raw', F.col('flight_type')).withColumn('flight_status_raw', F.col('flight_status'))
    d = _clean_text(d, ['flight_id', 'airline_id', 'airport_id', 'aircraft_id', 'timezone'])
    d = _clean_text(d, ['flight_type', 'flight_status', 'source_system'], upper=True)
    d = (d.withColumn('fare_amount', F.col('fare_amount').cast('decimal(12,2)'))
          .withColumn('delay_minutes', F.col('delay_minutes').cast('int'))
          .withColumn('tz_is_valid', F.coalesce(F.col('timezone').isin(VALID_TZ), F.lit(False)))
          .withColumn('tz_used', F.when(F.col('tz_is_valid'), F.col('timezone')).otherwise(F.lit(DEFAULT_SOURCE_TZ)))
          .withColumn('event_ts_utc', F.to_utc_timestamp('event_timestamp', F.col('tz_used')))
          .withColumn('ingest_ts_utc', F.to_utc_timestamp('ingest_timestamp', F.col('tz_used'))))

    a = (airline_keys.select(F.trim(F.col('airline_id')).alias('airline_id')).distinct()
         .withColumn('_airline_known', F.lit(True)))
    p = (airport_keys.select(F.trim(F.col('airport_id')).alias('airport_id')).distinct()
         .withColumn('_airport_known', F.lit(True)))
    d = d.join(F.broadcast(a), 'airline_id', 'left').join(F.broadcast(p), 'airport_id', 'left')

    checks = [
        ('DQ01', 'flight_id missing or not FLIGHT-nnnnnnn', F.coalesce(~F.col('flight_id').rlike('^FLIGHT-[0-9]{7}$'), F.lit(True))),
        ('DQ02', 'event_timestamp missing or invalid', F.col('event_timestamp').isNull()),
        ('DQ11', 'ingest_timestamp missing or invalid', F.col('ingest_timestamp').isNull()),
        ('DQ03', 'fare_amount missing, not numeric or negative', F.col('fare_amount').isNull() | (F.col('fare_amount') < 0)),
        ('DQ09', 'delay_minutes missing, not an integer or negative', F.col('delay_minutes').isNull() | (F.col('delay_minutes') < 0)),
        ('DQ10', 'flight_status missing or not an allowed value', F.coalesce(~F.col('flight_status').isin(ALLOWED_STATUS), F.lit(True))),
        ('DQ06', 'airline_id not found in airline reference', F.col('_airline_known').isNull()),
        ('DQ07', 'airport_id not found in airport reference', F.col('_airport_known').isNull()),
    ]
    return _first_failing_rule(d, checks)


def split_and_dedupe(classified):
    '''Returns (silver_df, rejected_df). Rejected = invalid rows + removed duplicates (audit).'''
    invalid = classified.filter(F.col('reject_rule').isNotNull())
    valid = classified.filter(F.col('reject_rule').isNull())

    # DQ08: identical content -> keep the copy from the earliest file
    w_exact = Window.partitionBy(*SOURCE_COLUMNS).orderBy(F.col('_source_file').asc(), F.col('_ingested_at').asc())
    valid = valid.withColumn('_rn_exact', F.row_number().over(w_exact))
    dup_exact = (valid.filter('_rn_exact > 1').withColumn('reject_rule', F.lit('DQ08'))
                 .withColumn('reject_reason', F.lit('Exact duplicate of an earlier copy (same flight_id, same content)')))
    valid = valid.filter('_rn_exact = 1')

    # DQ12: same flight_id, different content -> newest ingest time wins
    w_ver = Window.partitionBy('flight_id').orderBy(F.col('ingest_ts_utc').desc(), F.col('_source_file').asc())
    valid = valid.withColumn('_rn_ver', F.row_number().over(w_ver))
    dup_ver = (valid.filter('_rn_ver > 1').withColumn('reject_rule', F.lit('DQ12'))
               .withColumn('reject_reason', F.lit('Older version of a flight_id that has a newer record')))
    keep = valid.filter('_rn_ver = 1')

    silver = keep.select(
        'flight_id', 'event_timestamp', 'event_ts_utc', 'ingest_timestamp', 'ingest_ts_utc', 'airline_id', 'airport_id',
        F.coalesce(F.col('flight_type'), F.lit('UNKNOWN')).alias('flight_type'),
        F.col('flight_type').isNull().alias('flight_type_was_missing'),
        'flight_status',
        F.col('timezone').alias('timezone_raw'), 'tz_used', F.col('tz_is_valid').alias('timezone_is_valid'),
        'fare_amount', 'delay_minutes', 'aircraft_id', 'source_system', 'source_schema_version',
        (F.col('fare_amount') > F.lit(FARE_OUTLIER_THRESHOLD)).alias('is_fare_outlier'),
        (F.col('delay_minutes') > F.lit(DELAY_OUTLIER_THRESHOLD)).alias('is_delay_outlier'),
        'flight_type_raw', 'flight_status_raw', '_source_file', '_ingested_at',
        F.current_timestamp().alias('_processed_at'))

    rejected = (invalid.unionByName(dup_exact, allowMissingColumns=True).unionByName(dup_ver, allowMissingColumns=True)
                .select(F.lit('EVENTS').alias('stage'), F.col('reject_rule').alias('rule_id'),
                        F.col('reject_reason').alias('reason'), 'flight_id', F.col('_source_file').alias('source_file'),
                        F.to_json(F.struct(*SOURCE_COLUMNS)).alias('raw_record'),
                        F.current_timestamp().alias('rejected_at')))
    return silver, rejected


# ---------------- status CDC ----------------
def classify_cdc(cdc_df, known_flights_df):
    d = _clean_text(cdc_df, ['flight_id'])
    d = _clean_text(d, ['previous_status', 'new_status'], upper=True)
    d = (d.withColumn('source_sequence', F.col('source_sequence').cast('bigint'))
          .withColumn('change_ts_utc', F.to_utc_timestamp('change_timestamp', F.lit(DEFAULT_SOURCE_TZ))))
    known = (known_flights_df.select(F.trim(F.col('flight_id')).alias('flight_id')).distinct()
             .withColumn('_known', F.lit(True)))
    d = d.join(F.broadcast(known), 'flight_id', 'left')
    checks = [
        ('DQC1', 'flight_id missing', F.col('flight_id').isNull()),
        ('DQC2', 'change_timestamp missing or invalid', F.col('change_timestamp').isNull()),
        ('DQC3', 'source_sequence missing or not an integer', F.col('source_sequence').isNull()),
        ('DQC4', 'new_status not COMPLETED/REVERSED/CANCELLED', F.coalesce(~F.col('new_status').isin(CDC_ALLOWED_STATUS), F.lit(True))),
        ('DQC5', 'flight_id not found in silver_flight_events (orphan update)', F.col('_known').isNull()),
    ]
    return _first_failing_rule(d, checks)


def split_cdc(classified_cdc):
    '''Returns (valid_cdc, rejected_cdc). A repeated (flight_id, source_sequence) keeps the earliest file copy.'''
    invalid = classified_cdc.filter(F.col('reject_rule').isNotNull())
    valid = classified_cdc.filter(F.col('reject_rule').isNull())
    w = Window.partitionBy('flight_id', 'source_sequence').orderBy(F.col('_source_file').asc(), F.col('_ingested_at').asc())
    valid = valid.withColumn('_rn', F.row_number().over(w))
    dup = (valid.filter('_rn > 1').withColumn('reject_rule', F.lit('DQC6'))
           .withColumn('reject_reason', F.lit('Repeated flight_id + source_sequence')))
    valid = valid.filter('_rn = 1').drop('_rn')
    rejected = (invalid.unionByName(dup, allowMissingColumns=True)
                .select(F.lit('CDC').alias('stage'), F.col('reject_rule').alias('rule_id'), F.col('reject_reason').alias('reason'),
                        'flight_id', F.col('_source_file').alias('source_file'),
                        F.to_json(F.struct('flight_id', 'change_timestamp', 'previous_status', 'new_status',
                                           'change_reason', 'source_sequence')).alias('raw_record'),
                        F.current_timestamp().alias('rejected_at')))
    return valid, rejected


def latest_change_per_flight(valid_cdc):
    '''MERGE needs exactly one source row per flight: newest change (time, then sequence).'''
    w = Window.partitionBy('flight_id').orderBy(F.col('change_ts_utc').desc(), F.col('source_sequence').desc())
    return (valid_cdc.withColumn('_rn', F.row_number().over(w)).filter('_rn = 1')
            .select('flight_id', 'new_status', 'change_ts_utc', 'source_sequence'))


def build_history(events_df, cdc_valid):
    '''Original status of every flight (sequence 0) + every valid status change.'''
    initial = events_df.select(
        'flight_id', F.col('event_ts_utc').alias('effective_ts_utc'),
        F.lit(None).cast('string').alias('previous_status'), F.col('flight_status').alias('new_status'),
        F.lit('INITIAL_EVENT').alias('change_reason'), F.lit(0).cast('bigint').alias('source_sequence'),
        F.lit('INITIAL').alias('record_type'), F.lit(False).alias('is_noop'),
        F.col('_source_file').alias('source_file'), F.current_timestamp().alias('processed_at'))
    changes = cdc_valid.select(
        'flight_id', F.col('change_ts_utc').alias('effective_ts_utc'), 'previous_status', 'new_status', 'change_reason',
        F.col('source_sequence').cast('bigint').alias('source_sequence'), F.lit('CDC').alias('record_type'),
        F.coalesce(F.col('previous_status') == F.col('new_status'), F.lit(False)).alias('is_noop'),
        F.col('_source_file').alias('source_file'), F.current_timestamp().alias('processed_at'))
    return initial.unionByName(changes)


def status_as_of(events_df, history_df):
    '''Status in effect at each event time = newest history row with effective_ts_utc <= event_ts_utc.'''
    e = events_df.select('flight_id', 'event_ts_utc').alias('e')
    h = history_df.select('flight_id', 'effective_ts_utc', 'new_status', 'source_sequence').alias('h')
    j = e.join(h, (F.col('e.flight_id') == F.col('h.flight_id')) & (F.col('h.effective_ts_utc') <= F.col('e.event_ts_utc')), 'left')
    w = Window.partitionBy('e.flight_id', 'e.event_ts_utc').orderBy(F.col('h.effective_ts_utc').desc(), F.col('h.source_sequence').desc())
    return (j.withColumn('_rn', F.row_number().over(w)).filter('_rn = 1')
             .select(F.col('e.flight_id').alias('flight_id'), F.col('e.event_ts_utc').alias('event_ts_utc'),
                     F.col('h.new_status').alias('status_at_event_time'),
                     F.col('h.effective_ts_utc').alias('status_effective_ts_utc')))


# ---------------- secondary events ----------------
def build_secondary(sec_df, flights_df):
    '''Returns (silver_secondary, rejected_secondary).'''
    d = _clean_text(sec_df, ['event_id', 'flight_id', 'related_entity_id'])
    d = _clean_text(d, ['event_type', 'source_system'], upper=True)
    d = d.withColumn('event_ts_utc', F.to_utc_timestamp('event_timestamp', F.lit(DEFAULT_SOURCE_TZ)))
    known = (flights_df.select(F.trim(F.col('flight_id')).alias('flight_id')).distinct().withColumn('_known', F.lit(True)))
    d = d.join(F.broadcast(known), 'flight_id', 'left')
    d = _first_failing_rule(d, [
        ('DQS1', 'event_id missing', F.col('event_id').isNull()),
        ('DQS2', 'flight_id not found in silver_flight_events', F.col('_known').isNull())])
    invalid = d.filter(F.col('reject_rule').isNotNull())
    valid = d.filter(F.col('reject_rule').isNull())
    w = Window.partitionBy('event_id').orderBy(F.col('_source_file').asc(), F.col('_ingested_at').asc())
    valid = valid.withColumn('_rn', F.row_number().over(w))
    dup = (valid.filter('_rn > 1').withColumn('reject_rule', F.lit('DQS3')).withColumn('reject_reason', F.lit('Repeated event_id')))
    keep = valid.filter('_rn = 1')
    silver = keep.select('event_id', 'flight_id', 'event_timestamp', 'event_ts_utc', 'related_entity_id', 'event_type',
                         'event_value', 'source_system', '_source_file', '_ingested_at', F.current_timestamp().alias('_processed_at'))
    rejected = (invalid.unionByName(dup, allowMissingColumns=True)
                .select(F.lit('SECONDARY').alias('stage'), F.col('reject_rule').alias('rule_id'), F.col('reject_reason').alias('reason'),
                        'flight_id', F.col('_source_file').alias('source_file'),
                        F.to_json(F.struct('event_id', 'flight_id', 'event_timestamp', 'related_entity_id', 'event_type', 'event_value')).alias('raw_record'),
                        F.current_timestamp().alias('rejected_at')))
    return silver, rejected

In [0]:
# =====================================================
# DELTA TABLE HELPERS  (CREATE / MERGE - Databricks only)
# =====================================================
def ensure_tables():
    spark.sql(f'''CREATE TABLE IF NOT EXISTS {SILVER_REJECTED_TABLE} (
        stage STRING, rule_id STRING, reason STRING, flight_id STRING,
        source_file STRING, raw_record STRING, rejected_at TIMESTAMP) USING DELTA''')
    spark.sql(f'''CREATE TABLE IF NOT EXISTS {SILVER_FLIGHT_CURRENT} (
        flight_id STRING, airline_id STRING, airport_id STRING, flight_type STRING,
        event_ts_utc TIMESTAMP, fare_amount DECIMAL(12,2), delay_minutes INT, is_fare_outlier BOOLEAN,
        initial_status STRING, current_status STRING, status_updated_at TIMESTAMP,
        last_source_sequence BIGINT, status_source STRING) USING DELTA''')
    spark.sql(f'''CREATE TABLE IF NOT EXISTS {SILVER_STATUS_HISTORY} (
        flight_id STRING, effective_ts_utc TIMESTAMP, previous_status STRING, new_status STRING,
        change_reason STRING, source_sequence BIGINT, record_type STRING, is_noop BOOLEAN,
        source_file STRING, processed_at TIMESTAMP) USING DELTA''')


def replace_rejected(stage, rejected_df):
    '''Idempotent: wipe this stage's rejects, then write the fresh ones.'''
    spark.sql(f"DELETE FROM {SILVER_REJECTED_TABLE} WHERE stage = '{stage}'")
    (rejected_df.select('stage', 'rule_id', 'reason', 'flight_id', 'source_file', 'raw_record', 'rejected_at')
        .write.mode('append').saveAsTable(SILVER_REJECTED_TABLE))


def seed_current(target, events_df):
    '''Insert flights that are not in the current table yet. Never touches existing rows,
    so re-running Silver cannot erase status changes already applied by CDC.'''
    src = events_df.select('flight_id', 'airline_id', 'airport_id', 'flight_type', 'event_ts_utc', 'fare_amount',
                           'delay_minutes', 'is_fare_outlier',
                           F.col('flight_status').alias('initial_status'),
                           F.col('flight_status').alias('current_status'),
                           F.col('event_ts_utc').alias('status_updated_at'),
                           F.lit(0).cast('bigint').alias('last_source_sequence'),
                           F.lit('EVENT').alias('status_source'))
    src.createOrReplaceTempView('_seed_src')
    spark.sql(f'MERGE INTO {target} t USING _seed_src s ON t.flight_id = s.flight_id WHEN NOT MATCHED THEN INSERT *')


def append_history(target, history_df):
    '''Insert-only MERGE on (flight_id, source_sequence): replaying the same rows adds nothing.'''
    history_df.createOrReplaceTempView('_hist_src')
    spark.sql(f'MERGE INTO {target} t USING _hist_src s '
              f'ON t.flight_id = s.flight_id AND t.source_sequence = s.source_sequence '
              f'WHEN NOT MATCHED THEN INSERT *')


def merge_cdc(target, latest_df):
    '''Apply the newest change per flight, but only if it is newer than what the row already has.'''
    latest_df.createOrReplaceTempView('_cdc_latest')
    spark.sql(f'''
        MERGE INTO {target} t USING _cdc_latest s ON t.flight_id = s.flight_id
        WHEN MATCHED AND (s.change_ts_utc > t.status_updated_at
                          OR (s.change_ts_utc = t.status_updated_at AND s.source_sequence > t.last_source_sequence))
        THEN UPDATE SET t.current_status = s.new_status, t.status_updated_at = s.change_ts_utc,
                        t.last_source_sequence = s.source_sequence, t.status_source = 'CDC' ''')

print('Helpers defined.')

## Step A - Build `silver_flight_events` and its rejected rows

In [0]:
ensure_tables()

bronze_events = spark.table(BRONZE_FLIGHT_TABLE)
airline_keys = spark.table(AIRLINE_TABLE).select('airline_id')
airport_keys = spark.table(AIRPORT_TABLE).select('airport_id')

classified = classify_events(bronze_events, airline_keys, airport_keys)
silver_df, rejected_df = split_and_dedupe(classified)

(silver_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(SILVER_FLIGHT_TABLE))
replace_rejected('EVENTS', rejected_df)

n_silver = spark.table(SILVER_FLIGHT_TABLE).count()
n_rej = spark.table(SILVER_REJECTED_TABLE).filter(F.col('stage') == 'EVENTS').count()
print('silver_flight_events:', n_silver)
print('rejected (EVENTS):', n_rej)

In [0]:
# ---- Verification: nothing lost, keys unique, rules and flags as expected ----
b = spark.table(BRONZE_FLIGHT_TABLE).count()
s = spark.table(SILVER_FLIGHT_TABLE).count()
r = spark.table(SILVER_REJECTED_TABLE).filter(F.col('stage') == 'EVENTS').count()
print(b, '=', s, '+', r, '->', b == s + r)
assert b == s + r

dup_keys = spark.table(SILVER_FLIGHT_TABLE).groupBy('flight_id').count().filter('count > 1').count()
print('duplicate flight_id in silver:', dup_keys)
assert dup_keys == 0

print('Rejected rows by rule')
display(spark.table(SILVER_REJECTED_TABLE).groupBy('stage', 'rule_id', 'reason').count())

print('Flag counts')
display(spark.table(SILVER_FLIGHT_TABLE).agg(
    F.sum(F.col('is_fare_outlier').cast('int')).alias('fare_outliers'),
    F.sum(F.col('is_delay_outlier').cast('int')).alias('delay_outliers'),
    F.sum(F.col('flight_type_was_missing').cast('int')).alias('flight_type_filled'),
    F.sum((~F.col('timezone_is_valid')).cast('int')).alias('timezone_not_valid')))

print('One duplicate traced end to end: FLIGHT-0000501 (kept copy, then the removed copy)')
display(spark.table(SILVER_FLIGHT_TABLE).filter(F.col('flight_id') == 'FLIGHT-0000501')
        .select('flight_id', '_source_file', 'event_ts_utc', 'fare_amount', 'delay_minutes'))
display(spark.table(SILVER_REJECTED_TABLE).filter(F.col('flight_id') == 'FLIGHT-0000501'))

spark.table(SILVER_FLIGHT_TABLE).printSchema()
print('Status mix after de-duplication')
display(spark.table(SILVER_FLIGHT_TABLE).groupBy('flight_status').count().orderBy('flight_status'))

print('Before / after')
display(spark.createDataFrame([
    ('Bronze bronze_flight_events', b), ('Removed as exact duplicates (DQ08)', r),
    ('Silver silver_flight_events', s)], 'stage string, rows long'))

## Step B - Secondary events

In [0]:
secondary_bronze = spark.table(BRONZE_SECONDARY_TABLE)
silver_secondary_df, secondary_rejected_df = build_secondary(secondary_bronze, spark.table(SILVER_FLIGHT_TABLE))

(silver_secondary_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
    .saveAsTable(SILVER_SECONDARY_TABLE))
replace_rejected('SECONDARY', secondary_rejected_df)

print('Bronze secondary rows:', secondary_bronze.count())
print('silver_secondary_events:', spark.table(SILVER_SECONDARY_TABLE).count())
print('rejected (SECONDARY):', spark.table(SILVER_REJECTED_TABLE).filter(F.col('stage') == 'SECONDARY').count())
display(spark.table(SILVER_SECONDARY_TABLE).groupBy('event_type').count().orderBy('event_type'))

## Step C - Status CDC: current state, history and guarded MERGE

In [0]:
# C1. Seed the current-state table with EVERY flight (insert-only; existing rows are never touched)
seed_current(SILVER_FLIGHT_CURRENT, spark.table(SILVER_FLIGHT_TABLE))
print('silver_flight_current rows after seeding:', spark.table(SILVER_FLIGHT_CURRENT).count())
display(spark.table(SILVER_FLIGHT_CURRENT).groupBy('status_source').count())

# C2. Validate the CDC rows and quarantine the bad ones
cdc_classified = classify_cdc(spark.table(BRONZE_CDC_TABLE), spark.table(SILVER_FLIGHT_TABLE))
cdc_valid, cdc_rejected = split_cdc(cdc_classified)
replace_rejected('CDC', cdc_rejected)
print('CDC rows:', spark.table(BRONZE_CDC_TABLE).count(), '| valid:', cdc_valid.count(),
      '| rejected:', spark.table(SILVER_REJECTED_TABLE).filter(F.col('stage') == 'CDC').count())

# C3. History: original status of every flight + every valid change
append_history(SILVER_STATUS_HISTORY, build_history(spark.table(SILVER_FLIGHT_TABLE), cdc_valid))
print('silver_status_history rows:', spark.table(SILVER_STATUS_HISTORY).count())

# C4. Apply the newest change per flight with the guarded MERGE
latest = latest_change_per_flight(cdc_valid)
print('flights with a change to apply:', latest.count())
merge_cdc(SILVER_FLIGHT_CURRENT, latest)

print('Current status mix')
display(spark.table(SILVER_FLIGHT_CURRENT).groupBy('current_status').count().orderBy('current_status'))
print('Where the current status came from')
display(spark.table(SILVER_FLIGHT_CURRENT).groupBy('status_source').count())

In [0]:
# DELTA-ONLY: Delta history of the current table (screenshot for evidence; first MERGE should show 520 updated rows)
cur_hist = spark.sql(f'DESCRIBE HISTORY {SILVER_FLIGHT_CURRENT}')
display(cur_hist.select('version', 'operation', F.col('operationMetrics')['numTargetRowsInserted'].alias('rows_inserted'),
                        F.col('operationMetrics')['numTargetRowsUpdated'].alias('rows_updated')))

In [0]:
# ---- Verification of Step C ----
current = spark.table(SILVER_FLIGHT_CURRENT)
n_cur = current.count()
n_cur_distinct = current.select('flight_id').distinct().count()
print('current rows:', n_cur, '| distinct flights:', n_cur_distinct)
assert n_cur == n_cur_distinct == spark.table(SILVER_FLIGHT_TABLE).count() == 11500

print('rows changed by CDC:', current.filter(F.col('status_source') == 'CDC').count())
assert current.filter(F.col('status_source') == 'CDC').count() == 520

# every history row must be one of: original status (per flight) or a CDC change
h = spark.table(SILVER_STATUS_HISTORY)
print('history rows:', h.count(), '| initial:', h.filter(F.col('record_type') == 'INITIAL').count(),
      '| cdc:', h.filter(F.col('record_type') == 'CDC').count(),
      '| no-op changes:', h.filter(F.col('is_noop')).count())

# Trace one flight from raw CDC to current
fid = 'FLIGHT-0000001'
display(spark.table(BRONZE_CDC_TABLE).filter(F.col('flight_id') == fid))
display(h.filter(F.col('flight_id') == fid).orderBy('effective_ts_utc', 'source_sequence'))
display(current.filter(F.col('flight_id') == fid))

In [0]:
# ---- Replay test: running Step C again must change nothing ----
def snapshot(table):
    return spark.table(table).collect()

before_current = snapshot(SILVER_FLIGHT_CURRENT)
before_hist_n = spark.table(SILVER_STATUS_HISTORY).count()

seed_current(SILVER_FLIGHT_CURRENT, spark.table(SILVER_FLIGHT_TABLE))                       # must NOT reset CDC statuses
append_history(SILVER_STATUS_HISTORY, build_history(spark.table(SILVER_FLIGHT_TABLE), cdc_valid))
merge_cdc(SILVER_FLIGHT_CURRENT, latest_change_per_flight(cdc_valid))

after_current = snapshot(SILVER_FLIGHT_CURRENT)
print('current table unchanged after replay:', sorted(before_current) == sorted(after_current))
print('history rows before/after replay:', before_hist_n, spark.table(SILVER_STATUS_HISTORY).count())
assert sorted(before_current) == sorted(after_current)
assert before_hist_n == spark.table(SILVER_STATUS_HISTORY).count()
print('PASS - replay is safe')

## Step D - Status at event time (as-of join) and reconciliation

In [0]:
asof = status_as_of(spark.table(SILVER_FLIGHT_TABLE), spark.table(SILVER_STATUS_HISTORY))
(asof.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(SILVER_ASOF_TABLE))

chk = (spark.table(SILVER_ASOF_TABLE)
       .join(spark.table(SILVER_FLIGHT_TABLE).select('flight_id', 'flight_status'), 'flight_id'))
print('as-of rows:', chk.count(), '| status at event time equals reported status:',
      chk.filter(F.col('status_at_event_time') == F.col('flight_status')).count())
# Every CDC change is dated 2026-08-10, after the last flight event (2026-08-01), so these must match.
# The join logic itself is proven by unit test U10 with a change BEFORE the event.

print('Silver table row counts')
for name in [SILVER_FLIGHT_TABLE, SILVER_REJECTED_TABLE, SILVER_STATUS_HISTORY, SILVER_FLIGHT_CURRENT,
             SILVER_SECONDARY_TABLE, SILVER_ASOF_TABLE, DQ_TABLE]:
    print(f'{name.split(".")[-1]:36s}', spark.table(name).count())

## Step E - Unit tests (they call the same functions the pipeline uses)

In [0]:
import datetime as dt

BRONZE_TEST_SCHEMA = ('flight_id string, event_timestamp timestamp, ingest_timestamp timestamp, airline_id string, '
                      'airport_id string, flight_type string, flight_status string, timezone string, fare_amount double, '
                      'delay_minutes int, aircraft_id string, source_system string, source_schema_version string, '
                      '_rescued_data string, _source_file string, _ingested_at timestamp')
TEST_COLS = [c.split(' ')[0] for c in BRONZE_TEST_SCHEMA.split(', ')]


def ev(**kw):
    base = dict(flight_id='FLIGHT-0000001', event_timestamp=dt.datetime(2026, 7, 1, 10, 0, 0),
                ingest_timestamp=dt.datetime(2026, 7, 1, 10, 5, 0), airline_id='AIRLINE_ID-0001',
                airport_id='AIRPORT_ID-0001', flight_type='DOMESTIC', flight_status='OPEN', timezone='USD',
                fare_amount=500.0, delay_minutes=20, aircraft_id='AIRCRAFT_ID-0001', source_system='ADLS_EVENT',
                source_schema_version=None, _rescued_data=None, _source_file='file_a.csv',
                _ingested_at=dt.datetime(2026, 8, 1, 0, 0, 0))
    base.update(kw)
    return tuple(base[c] for c in TEST_COLS)


TEST_AIRLINES = spark.createDataFrame([('AIRLINE_ID-0001',)], 'airline_id string')
TEST_AIRPORTS = spark.createDataFrame([('AIRPORT_ID-0001',)], 'airport_id string')


def run_events(rows):
    df = spark.createDataFrame(rows, BRONZE_TEST_SCHEMA)
    return split_and_dedupe(classify_events(df, TEST_AIRLINES, TEST_AIRPORTS))


CDC_TEST_SCHEMA = ('flight_id string, change_timestamp timestamp, previous_status string, new_status string, '
                   'change_reason string, source_sequence int, source_system string, _source_file string, _ingested_at timestamp')


def cdc_row(flight_id, ts, new_status, seq, f='cdc_a.csv', prev='OPEN'):
    return (flight_id, ts, prev, new_status, 'TEST', seq, 'ADLS_CDC', f, dt.datetime(2026, 8, 11, 0, 0, 0))


test_results = []


def check(test_id, description, condition):
    test_results.append((test_id, description, 'PASS' if condition else 'FAIL'))


check('U00', 'time-zone list is available on this compute', 'Asia/Kolkata' in VALID_TZ)

# U01 valid row; real time zone converts to UTC
s_, r_ = run_events([ev(timezone='Asia/Kolkata')])
check('U01', 'valid row kept; 10:00 Asia/Kolkata becomes 04:30 UTC', r_.count() == 0 and str(s_.first().event_ts_utc) == '2026-07-01 04:30:00')

# U02 every reject rule routes to the right rule ID
bad = [ev(flight_id=None),
       ev(flight_id='BADKEY'),
       ev(flight_id='FLIGHT-0000002', event_timestamp=None),
       ev(flight_id='FLIGHT-0000003', ingest_timestamp=None),
       ev(flight_id='FLIGHT-0000004', fare_amount=None),
       ev(flight_id='FLIGHT-0000005', fare_amount=-1.0),
       ev(flight_id='FLIGHT-0000006', delay_minutes=None),
       ev(flight_id='FLIGHT-0000007', delay_minutes=-5),
       ev(flight_id='FLIGHT-0000008', flight_status='LANDED'),
       ev(flight_id='FLIGHT-0000009', airline_id='AIRLINE_ID-9999'),
       ev(flight_id='FLIGHT-0000010', airport_id='AIRPORT_ID-9999')]
s_, r_ = run_events(bad)
got = {x.flight_id: x.rule_id for x in r_.collect()}
expected = {None: 'DQ01', 'BADKEY': 'DQ01', 'FLIGHT-0000002': 'DQ02', 'FLIGHT-0000003': 'DQ11', 'FLIGHT-0000004': 'DQ03',
            'FLIGHT-0000005': 'DQ03', 'FLIGHT-0000006': 'DQ09', 'FLIGHT-0000007': 'DQ09', 'FLIGHT-0000008': 'DQ10',
            'FLIGHT-0000009': 'DQ06', 'FLIGHT-0000010': 'DQ07'}
check('U02', 'invalid rows are rejected with the correct rule ID and none reach Silver', s_.count() == 0 and got == expected)

# U03 duplicates: exact copies (DQ08) and a changed version (DQ12)
rows = [ev(flight_id='FLIGHT-0000020', _source_file='file_b.csv'), ev(flight_id='FLIGHT-0000020', _source_file='file_a.csv'),
        ev(flight_id='FLIGHT-0000021'),
        ev(flight_id='FLIGHT-0000021', delay_minutes=99, ingest_timestamp=dt.datetime(2026, 7, 1, 11, 0, 0))]
s_, r_ = run_events(rows)
check('U03a', 'exact duplicate: earliest file survives, other copy audited as DQ08',
      s_.filter(F.col('flight_id') == 'FLIGHT-0000020').first()._source_file == 'file_a.csv'
      and [x.rule_id for x in r_.filter(F.col('flight_id') == 'FLIGHT-0000020').collect()] == ['DQ08'])
check('U03b', 'changed version: newest ingest time survives, older audited as DQ12',
      s_.filter(F.col('flight_id') == 'FLIGHT-0000021').first().delay_minutes == 99
      and [x.rule_id for x in r_.filter(F.col('flight_id') == 'FLIGHT-0000021').collect()] == ['DQ12'])

# U04-U06 flags and fills keep the row
s_, r_ = run_events([ev(timezone='USD', flight_type='', fare_amount=77083.5, delay_minutes=200)])
x = s_.first()
check('U04', 'USD is not a time zone: assume UTC and flag it', x.tz_used == 'UTC' and x.timezone_is_valid is False)
check('U05', 'blank flight_type becomes UNKNOWN and is flagged', x.flight_type == 'UNKNOWN' and x.flight_type_was_missing)
check('U06', 'extreme fare and delay are kept and flagged, not rejected', x.is_fare_outlier and x.is_delay_outlier and r_.count() == 0)

# U07 CDC validation rules
known = run_events([ev()])[0].select('flight_id')
t0 = dt.datetime(2026, 8, 10, 0, 5, 0)
cdc_test = spark.createDataFrame([
    cdc_row('FLIGHT-0000001', t0, 'CANCELLED', 2),
    cdc_row('FLIGHT-9999999', t0, 'CANCELLED', 3),
    cdc_row('FLIGHT-0000001', None, 'CANCELLED', 4),
    cdc_row('FLIGHT-0000001', t0, 'LANDED', 5),
    cdc_row('FLIGHT-0000001', t0, 'CANCELLED', None),
    cdc_row('FLIGHT-0000001', t0, 'CANCELLED', 2, f='cdc_b.csv')], CDC_TEST_SCHEMA)
valid_, rejected_ = split_cdc(classify_cdc(cdc_test, known))
got = sorted([(x.rule_id, x.source_file) for x in rejected_.collect()])
check('U07', 'CDC: orphan, bad time, bad status, missing sequence and repeated sequence are rejected; one valid row remains',
      valid_.count() == 1 and got == sorted([('DQC5', 'cdc_a.csv'), ('DQC2', 'cdc_a.csv'), ('DQC4', 'cdc_a.csv'),
                                             ('DQC3', 'cdc_a.csv'), ('DQC6', 'cdc_b.csv')]))

# U08 out-of-order batch: newest time wins; equal time -> higher sequence wins
batch = spark.createDataFrame([
    cdc_row('FLIGHT-0000001', dt.datetime(2026, 8, 10, 0, 5, 0), 'CANCELLED', 2),
    cdc_row('FLIGHT-0000001', dt.datetime(2026, 8, 10, 0, 1, 0), 'COMPLETED', 1),
    cdc_row('FLIGHT-0000001', dt.datetime(2026, 8, 10, 0, 5, 0), 'REVERSED', 9)], CDC_TEST_SCHEMA)
v_, _ = split_cdc(classify_cdc(batch, known))
lat = latest_change_per_flight(v_)
check('U08', 'out-of-order batch: one deterministic latest change per flight', lat.count() == 1 and lat.first().new_status == 'REVERSED' and lat.first().source_sequence == 9)

# U10 as-of join: a change BEFORE the event time must win; a change AFTER must not
ev_df = spark.createDataFrame([('F1', dt.datetime(2026, 7, 1, 12, 0, 0))], 'flight_id string, event_ts_utc timestamp')
hist_df = spark.createDataFrame([('F1', dt.datetime(2026, 7, 1, 8, 0, 0), 'OPEN', 0), ('F1', dt.datetime(2026, 7, 1, 10, 0, 0), 'CANCELLED', 5),
                                 ('F1', dt.datetime(2026, 7, 1, 14, 0, 0), 'COMPLETED', 9)],
                                'flight_id string, effective_ts_utc timestamp, new_status string, source_sequence bigint')
check('U10', 'as-of join returns the status in effect at event time (CANCELLED)', status_as_of(ev_df, hist_df).first().status_at_event_time == 'CANCELLED')

# U15 secondary events: orphan flight and repeated event_id
sec_test = spark.createDataFrame([
    ('E1', 'FLIGHT-0000001', dt.datetime(2026, 7, 1, 10, 0, 0), 'AIRLINE_ID-0001', 'CHECK', 1.0, 'ADLS_AUX', 'a.csv', dt.datetime(2026, 8, 1)),
    ('E1', 'FLIGHT-0000001', dt.datetime(2026, 7, 1, 10, 0, 0), 'AIRLINE_ID-0001', 'CHECK', 1.0, 'ADLS_AUX', 'b.csv', dt.datetime(2026, 8, 1)),
    ('E2', 'FLIGHT-9999999', dt.datetime(2026, 7, 1, 10, 0, 0), 'AIRLINE_ID-0001', 'ALERT', 2.0, 'ADLS_AUX', 'a.csv', dt.datetime(2026, 8, 1)),
    (None, 'FLIGHT-0000001', dt.datetime(2026, 7, 1, 10, 0, 0), 'AIRLINE_ID-0001', 'ALERT', 2.0, 'ADLS_AUX', 'a.csv', dt.datetime(2026, 8, 1))],
    'event_id string, flight_id string, event_timestamp timestamp, related_entity_id string, event_type string, event_value double, '
    'source_system string, _source_file string, _ingested_at timestamp')
s_sec, r_sec = build_secondary(sec_test, known)
check('U15', 'secondary events: missing id, orphan flight and repeated id are rejected; one valid event remains',
      s_sec.count() == 1 and sorted([x.rule_id for x in r_sec.collect()]) == ['DQS1', 'DQS2', 'DQS3'])
print('cleaning / CDC / as-of tests done:', len(test_results))

In [0]:
# ---- MERGE tests on temporary Delta tables (use the real merge functions) ----
test_cur = f'{CATALOG}.{SILVER_SCHEMA}.tmp_test_current'
test_hist = f'{CATALOG}.{SILVER_SCHEMA}.tmp_test_history'
try:
    for name in [test_cur, test_hist]:
        spark.sql(f'DROP TABLE IF EXISTS {name}')
    spark.sql(f'CREATE TABLE {test_cur} AS SELECT * FROM {SILVER_FLIGHT_CURRENT} WHERE 1 = 0')
    spark.sql(f'CREATE TABLE {test_hist} AS SELECT * FROM {SILVER_STATUS_HISTORY} WHERE 1 = 0')

    events_one = run_events([ev(flight_status='OPEN')])[0]          # FLIGHT-0000001, OPEN at 2026-07-01 10:00 UTC
    seed_current(test_cur, events_one)

    def apply_cdc(rows):
        c = classify_cdc(spark.createDataFrame(rows, CDC_TEST_SCHEMA), spark.table(test_cur).select('flight_id'))
        v, _ = split_cdc(c)
        merge_cdc(test_cur, latest_change_per_flight(v))

    def status_now():
        return spark.table(test_cur).first().current_status

    newer = [cdc_row('FLIGHT-0000001', dt.datetime(2026, 8, 10, 0, 5, 0), 'CANCELLED', 2)]
    late = [cdc_row('FLIGHT-0000001', dt.datetime(2026, 8, 10, 0, 1, 0), 'COMPLETED', 1)]

    apply_cdc(newer)
    check('U11a', 'a later status revision is applied', status_now() == 'CANCELLED')
    apply_cdc(late)
    check('U11b', 'a LATE (older) update does not overwrite the newer status', status_now() == 'CANCELLED')

    before = spark.table(test_cur).collect()
    apply_cdc(newer)
    apply_cdc(late)
    check('U12', 'replaying the same updates changes nothing', sorted(spark.table(test_cur).collect()) == sorted(before))

    seed_current(test_cur, events_one)
    check('U14', 're-seeding Silver does not erase a status already applied by CDC', status_now() == 'CANCELLED')

    h = spark.createDataFrame(
        [('FLIGHT-0000001', dt.datetime(2026, 8, 10, 0, 5, 0), 'OPEN', 'CANCELLED', 'X', 2, 'CDC', False, 'cdc_test.csv', dt.datetime(2026, 8, 11)),
         ('FLIGHT-0000001', dt.datetime(2026, 8, 10, 0, 1, 0), 'OPEN', 'COMPLETED', 'X', 1, 'CDC', False, 'cdc_test.csv', dt.datetime(2026, 8, 11))],
        'flight_id string, effective_ts_utc timestamp, previous_status string, new_status string, change_reason string, '
        'source_sequence bigint, record_type string, is_noop boolean, source_file string, processed_at timestamp')
    append_history(test_hist, h)
    append_history(test_hist, h)
    check('U13', 'history keeps the late update too, and a replay adds no duplicate rows', spark.table(test_hist).count() == 2)
finally:
    for name in [test_cur, test_hist]:
        spark.sql(f'DROP TABLE IF EXISTS {name}')

display(spark.createDataFrame(test_results, 'test_id string, description string, result string').orderBy('test_id'))
failed = [r for r in test_results if r[2] != 'PASS']
assert not failed, f'{len(failed)} test(s) failed: {failed}'
print('ALL', len(test_results), 'TESTS PASSED')

## Step F - Final validation (read-only)

In [0]:
# ============================================================
# DAY 2 FINAL VALIDATION - read-only checks of Bronze, Silver, DQ and CDC
# ============================================================
vresults = []

def vcheck(name, condition, detail=''):
    vresults.append((name, 'PASS' if condition else 'FAIL', detail))
    print(('PASS' if condition else 'FAIL') + ' | ' + name + (f' | {detail}' if detail else ''))

def vinfo(name, detail):
    vresults.append((name, 'INFO', detail))
    print(f'INFO | {name} | {detail}')

for name in [BRONZE_FLIGHT_TABLE, BRONZE_CDC_TABLE, BRONZE_SECONDARY_TABLE, AIRLINE_TABLE, AIRPORT_TABLE] + ALL_SILVER_TABLES:
    vcheck('Table exists: ' + name.split('.')[-1], spark.catalog.tableExists(name))

bf, bc, bs = spark.table(BRONZE_FLIGHT_TABLE), spark.table(BRONZE_CDC_TABLE), spark.table(BRONZE_SECONDARY_TABLE)
sf, rej, hist, cur = (spark.table(SILVER_FLIGHT_TABLE), spark.table(SILVER_REJECTED_TABLE),
                      spark.table(SILVER_STATUS_HISTORY), spark.table(SILVER_FLIGHT_CURRENT))
sec, asof_t = spark.table(SILVER_SECONDARY_TABLE), spark.table(SILVER_ASOF_TABLE)

# counts and reconciliation
vcheck('Silver flight events = 11,500', sf.count() == 11500, f'actual {sf.count()}')
rej_events = rej.filter(F.col('stage') == 'EVENTS')
vcheck('Bronze flights = Silver + rejected', bf.count() == sf.count() + rej_events.count(), f'{bf.count()} = {sf.count()} + {rej_events.count()}')
vcheck('Duplicates audited as DQ08 = 40', rej_events.filter(F.col('rule_id') == 'DQ08').count() == 40)
vcheck('Rejected rows carry rule_id, reason and source_file',
       rej.filter(F.col('rule_id').isNull() | F.col('reason').isNull() | F.col('source_file').isNull()).count() == 0)
vcheck('Flight_id is unique in Silver', sf.select('flight_id').distinct().count() == sf.count())
vcheck('No exact source duplicates in Silver', sf.count() == sf.select('flight_id', 'event_timestamp', 'airline_id', 'airport_id', 'fare_amount', 'delay_minutes').distinct().count())

# flags
vcheck('flight_type blanks flagged = 12', sf.filter(F.col('flight_type_was_missing')).count() == 12)
vcheck('flight_type never NULL in Silver', sf.filter(F.col('flight_type').isNull()).count() == 0)
vcheck('Fare outliers flagged = 35', sf.filter(F.col('is_fare_outlier')).count() == 35)
vcheck('Outlier rows kept (not rejected)', rej_events.filter(F.col('rule_id') == 'DQ05').count() == 0)
vcheck('Every row has a UTC event time', sf.filter(F.col('event_ts_utc').isNull()).count() == 0)

# reference integrity
unmatched_airline = sf.join(spark.table(AIRLINE_TABLE).select('airline_id'), 'airline_id', 'left_anti').count()
unmatched_airport = sf.join(spark.table(AIRPORT_TABLE).select('airport_id'), 'airport_id', 'left_anti').count()
vcheck('Airline reference integrity', unmatched_airline == 0, f'unmatched {unmatched_airline}')
vcheck('Airport reference integrity', unmatched_airport == 0, f'unmatched {unmatched_airport}')

# Bronze hygiene
vcheck('No rescued Bronze records', bf.filter(F.col('_rescued_data').isNotNull()).count() == 0 if '_rescued_data' in bf.columns else True)
vcheck('Bronze lineage columns populated', bf.filter(F.col('_source_file').isNull() | F.col('_ingested_at').isNull()).count() == 0)

# CDC
vcheck('CDC rows = 520', bc.count() == 520)
vcheck('No rejected CDC rows', rej.filter(F.col('stage') == 'CDC').count() == 0)
vcheck('Current table has one row per flight (11,500)', cur.count() == 11500 and cur.select('flight_id').distinct().count() == 11500)
vcheck('Current covers every Silver flight', sf.join(cur.select('flight_id'), 'flight_id', 'left_anti').count() == 0)
vcheck('Exactly 520 flights changed by CDC', cur.filter(F.col('status_source') == 'CDC').count() == 520)
vcheck('History = 11,500 initial + 520 changes', hist.count() == 12020 and hist.filter(F.col('record_type') == 'CDC').count() == 520)
vcheck('No repeated (flight_id, source_sequence) in history', hist.groupBy('flight_id', 'source_sequence').count().filter('count > 1').count() == 0)
latest_hist = (hist.groupBy('flight_id').agg(F.max('source_sequence').alias('max_seq')))
wrong = cur.join(latest_hist, 'flight_id').filter(F.col('last_source_sequence') != F.col('max_seq')).count()
vcheck('Current uses the latest sequence for every flight', wrong == 0, f'wrong rows {wrong}')
status_counts = {r['current_status']: r['count'] for r in cur.groupBy('current_status').count().collect()}
vcheck('Current status mix as expected', status_counts == {'CANCELLED': 171, 'COMPLETED': 2890, 'FAILED': 2836, 'OPEN': 2740, 'PENDING': 2690, 'REVERSED': 173}, str(status_counts))
vcheck('As-of table has one row per flight event', asof_t.count() == 11500)

# secondary
vcheck('Secondary silver = 310 and event_id unique', sec.count() == 310 and sec.select('event_id').distinct().count() == 310)
vcheck('Secondary events all link to Silver flights', sec.join(sf.select('flight_id'), 'flight_id', 'left_anti').count() == 0)

# catalogue and tests
vcheck('DQ catalogue has 23 rules', spark.table(DQ_TABLE).count() == 23)
vcheck('Unit tests all passed', len(test_results) > 0 and all(r[2] == 'PASS' for r in test_results), f'{len(test_results)} tests')

vinfo('Timezone', 'timezone column holds USD in every row; timestamps assumed UTC and flagged (timezone_is_valid = false).')
vinfo('Schedule revisions', 'No schedule dataset supplied; the effective-time join runs on status revisions instead.')
vinfo('REVERSED status', 'Kept exactly as received; business meaning not documented by the source.')

passed = sum(1 for r in vresults if r[1] == 'PASS')
failed = sum(1 for r in vresults if r[1] == 'FAIL')
print('=' * 65)
print('DAY 2 FINAL VALIDATION SUMMARY | passed:', passed, '| failed:', failed, '| info:', sum(1 for r in vresults if r[1] == 'INFO'))
print('RESULT:', 'ALL EXECUTED VALIDATION CHECKS PASSED' if failed == 0 else 'VALIDATION NEEDS ATTENTION')
print('=' * 65)
for name, status, detail in vresults:
    if status == 'FAIL':
        print('-', name, detail)
assert failed == 0

In [0]:
%sql
SELECT
    rule_id,
    reason,
    COUNT(*) AS record_count
FROM capstone_3flight.silver_schema.silver_rejected_records
GROUP BY rule_id, reason
ORDER BY rule_id;

In [0]:
%sql
DESCRIBE TABLE
capstone_3flight.silver_schema.silver_rejected_records;

# Day 2 - Silver layer: decisions and limitations

1. **Timezone:** the `timezone` column holds `USD` in every row, which is a currency, not a time zone. Timestamps are therefore **assumed to be UTC**. Every row is flagged `timezone_is_valid = false`, and the code converts correctly if a real zone name ever appears (unit test U01).
2. **Schedule revisions:** no schedule dataset was supplied. The effective-time (as-of) join is implemented on **status revisions** from the CDC feed. Every CDC change is dated 2026-08-10, after the last flight event (2026-08-01), so the status at event time equals the reported status for all flights; the logic is proven by unit test U10.
3. **CDC late-arrival:** the supplied CDC file has no late or out-of-order rows. Late, out-of-order, replayed and repeated-sequence cases are covered by unit tests U07, U08, U11 to U14 that call the production functions.
4. **CDC ordering:** the newest change wins by `change_timestamp`, then `source_sequence`. In this data both orderings agree.
5. **Duplicates:** all 40 duplicates are exact copies. The copy from the earliest file is kept; each removed copy is written to `silver_rejected_records` under rule `DQ08`, so Bronze = Silver + rejected.
6. **Extreme fares:** 35 fares above 5,000 (normal maximum 1,099.99) are **flagged** (`is_fare_outlier`), never rejected. The threshold is a data-quality flag and still needs client approval.
7. **REVERSED:** appears as a CDC status (173 flights). Its business meaning is not documented, so it is kept as received and not counted as cancelled.
8. **Bronze typing:** Bronze stores timestamps, fares and delays as typed columns. A value that cannot be cast would be NULL or in `_rescued_data` before Silver sees it; Silver rejects NULLs under DQ02, DQ03, DQ09 and DQ11, and the rescued-data count is verified as 0.
9. **Rebuild pattern:** `silver_flight_events` is rebuilt from Bronze on every run (idempotent). `silver_flight_current` is seeded insert-only and changed only through the guarded MERGE, so rebuilding events cannot erase applied statuses.
10. **Inactive references:** 6 airlines and 8 airports are marked inactive in the reference data and are used by flights. They are not rejected; this is an observation, not a rule.